# SWAP-Stress: the 9 km static covariate rasters

A visual tour of the static covariates the model reads at inference time.

Every raster here was exported from Earth Engine at ~9 km and reprojected onto
the SMAP EASE-Grid 2.0 window by `swapstress.features.reproject_to_ease2`, so it
aligns pixel for pixel with the daily SMAP L3 soil moisture GeoTIFFs. That
alignment is the whole point: stage 05 reads one column of covariates per pixel
with no resampling.

1. Which rasters exist, and how many bands each carries
2. Representative maps per feature group
3. Band counts across groups

The stack the released model reads is **pruned** — it carries only the groups
that survived feature selection. Section 2 walks every group that was ever
extracted; the ones absent from whatever stack `FEAT_DIR` points at say so and
move on.

**Outputs are not committed.** Rendered maps land in `notebooks/_outputs/`;
clear cell outputs before committing this notebook.

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from swapstress.config import load_config
from swapstress.features.smap_download import resolve_ease2_grid
from swapstress.figures import basemap

# ---------------------------------------------------------------------------
# The raster directory comes from the same run config stage 05 reads, so this
# notebook always tours the stack the released model actually predicts from.
# REPO is the only thing you may need to change.
#
# That stack is pruned: it carries only the feature groups the released model
# kept. Every other group below will report itself absent. Point FEAT_DIR at a
# fuller stack — the CONUS feature directory carries POLARIS, PRISM, SSURGO,
# terrain, Sentinel-1, land cover, and the SMAP L3 climatology as well — and the
# same cells will draw them.
# ---------------------------------------------------------------------------
REPO = os.path.abspath(os.environ.get("SWAPSTRESS_REPO", "."))

predict_cfg = load_config(
    os.path.join(REPO, "configs", "predict_9km_global_pruned.toml"), {}
)
FEAT_DIR = os.environ.get("SWAPSTRESS_FEATURE_DIR", predict_cfg["static_dir"])

OUT_DIR = os.path.join("notebooks", "_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# The CONUS window, so a CONUS raster can be recognized and given state outlines.
_conus_rows, _conus_cols, _ = resolve_ease2_grid("conus")
CONUS_SHAPE = (
    _conus_rows.stop - _conus_rows.start,
    _conus_cols.stop - _conus_cols.start,
)

tifs = sorted(f for f in os.listdir(FEAT_DIR) if f.endswith("_ease2.tif"))
print("FEAT_DIR:  ", FEAT_DIR)
print("EASE2 files:", len(tifs))
print("CONUS window shape (rows, cols):", CONUS_SHAPE)

## 1) Inventory

Each file is one feature group. `swapstress.features.ee_feature_list.label_feature`
turns a band name into a human-readable description, and
`swapstress.features.features.classify_feature` assigns it to the same group the
model's feature selection and importance aggregation use.

In [ ]:
from swapstress.features.ee_feature_list import label_feature
from swapstress.features.features import classify_feature

records = []
for fname in tifs:
    with rasterio.open(os.path.join(FEAT_DIR, fname)) as src:
        descs = [src.descriptions[i] or f"band_{i + 1}" for i in range(src.count)]
        records.append(
            {
                "file": fname,
                "bands": src.count,
                "shape": f"{src.width}x{src.height}",
                "dtype": str(src.dtypes[0]),
                "crs": str(src.crs),
                "first_bands": ", ".join(descs[:4]) + ("..." if src.count > 4 else ""),
            }
        )

inv = pd.DataFrame(records)
print(f"{len(inv)} files, {inv['bands'].sum()} bands total\n")
inv

In [ ]:
# Every band, with its label and its model-side group.
band_rows = []
for fname in tifs:
    with rasterio.open(os.path.join(FEAT_DIR, fname)) as src:
        for i in range(src.count):
            name = src.descriptions[i] or f"band_{i + 1}"
            band_rows.append(
                {
                    "file": fname,
                    "band_index": i + 1,
                    "band": name,
                    "label": label_feature(name),
                    "group": classify_feature(name),
                }
            )

bands_df = pd.DataFrame(band_rows)
print(f"{len(bands_df)} bands across {bands_df['file'].nunique()} files")
bands_df.head(20)

In [ ]:
def plot_group(stem, band_names, ncols=2, cmap="viridis", title=None):
    """Draw named bands from <stem>_ease2.tif as a grid of maps.

    Notebook display plumbing only: it reads bands and calls imshow. CONUS-window
    rasters get state outlines from swapstress.figures.basemap; a global raster
    is drawn bare, since CONUS outlines would be meaningless over it.
    """
    path = os.path.join(FEAT_DIR, f"{stem}_ease2.tif")
    if not os.path.exists(path):
        print(f"{stem}: not in this stack ({os.path.basename(FEAT_DIR)})")
        return None

    with rasterio.open(path) as src:
        descs = [src.descriptions[i] or f"band_{i + 1}" for i in range(src.count)]
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
        is_conus = (src.height, src.width) == CONUS_SHAPE
        epsg = src.crs.to_epsg()
        picks = [(n, descs.index(n)) for n in band_names if n in descs]
        if not picks:
            print(f"{stem}: none of {band_names} present; available: {descs[:12]}")
            return None
        arrays = [(n, src.read(i + 1)) for n, i in picks]

    states = basemap.load_conus_states(crs=epsg) if is_conus else None

    nrows = (len(arrays) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), dpi=110)
    axes = np.atleast_1d(axes).ravel()

    for ax, (name, arr) in zip(axes, arrays):
        valid = arr[np.isfinite(arr)]
        vmin, vmax = np.percentile(valid, [2, 98]) if valid.size else (0, 1)
        im = ax.imshow(
            arr, extent=extent, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax
        )
        if states is not None:
            states.boundary.plot(ax=ax, color="0.3", linewidth=0.3)
            ax.set_xlim(extent[0], extent[1])
            ax.set_ylim(extent[2], extent[3])
        fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
        ax.set_title(f"{name}\n{label_feature(name)}", fontsize=9)
        ax.tick_params(labelsize=7)

    for ax in axes[len(arrays) :]:
        ax.set_visible(False)

    if title:
        fig.suptitle(title, fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()
    return fig


print("plot_group ready")

## 2) The feature groups

### 2a) SoilGrids (ISRIC, 250 m native)

Global machine-learned predictions of soil properties from ~240k profiles. Two
depth intervals (0-5 cm, 5-15 cm) × 10 properties: bulk density, CEC, coarse
fragments, clay, sand, silt, nitrogen, pH, SOC, and organic carbon density.

In [ ]:
plot_group(
    "soilgrids",
    ["clay_0-5cm_mean", "sand_0-5cm_mean", "soc_0-5cm_mean", "bdod_0-5cm_mean"],
    cmap="YlOrBr",
    title="SoilGrids",
)

### 2b) WorldClim bioclimatic normals + reference ETo (1 km native)

WorldClim v2.1 normals (1970-2000) reduced to seasonal means — precipitation,
tavg, tmin, tmax — alongside MODIS-based reference evapotranspiration seasonal
means and annual statistics.

In [ ]:
plot_group(
    "worldclim",
    ["wc_prec_summer", "wc_tavg_summer", "eto_yearly_mean"],
    cmap="coolwarm",
    title="WorldClim + reference ETo",
)

### 2c) SMAP L3 vegetation water content climatology (9 km native)

Multi-year mean and standard deviation of the vegetation water content retrieved
alongside soil moisture, AM and PM passes separately — a proxy for canopy water
status and vegetation density.

This is an **L3** retrieval, not an L4 model state. L4 is never used as a
feature: it assimilates brightness temperature into a land surface model that
carries its own pedotransfer hydraulic parameters, so using it would leak those
assumptions into a model built to replace them.

In [ ]:
plot_group(
    "smap_l3_clim",
    ["vegetation_water_content_am_mean", "vegetation_water_content_am_stdDev"],
    cmap="YlGn",
    title="SMAP L3 vegetation water content climatology",
)

### 2d) Terrain (SRTM 30 m native, lithology 1 km)

SRTM-derived elevation, slope, aspect, and topographic position index at two
neighborhood scales (10 km and 22.5 km), plus lithology and a constant band.

In [ ]:
plot_group(
    "terrain", ["elevation", "slope", "tpi_10000"], cmap="terrain", title="Terrain"
)

### 2e) POLARIS soil properties (30 m native, CONUS only)

Probabilistic soil property maps for CONUS from SSURGO plus machine learning:
texture, bulk density, saturated hydraulic conductivity, porosity, and van
Genuchten / Brooks-Corey parameters. POLARIS is also one of the pedotransfer
baselines the technical validation contrasts the model against — see
`swapstress.validation.ptf_baseline`.

In [ ]:
plot_group(
    "polaris",
    ["clay_mean", "ksat_mean", "theta_s_mean", "alpha_mean"],
    cmap="YlOrBr",
    title="POLARIS",
)

### 2f) PRISM climate normals (800 m native, CONUS only)

PRISM 30-year normals (1991-2020): precipitation, temperature (mean, min, max),
dew point, VPD (min, max), and solar radiation. Finer spatial detail than
WorldClim, but CONUS only — which is why the global release leans on WorldClim.

In [ ]:
plot_group(
    "prism_normals", ["ppt", "tmean", "vpdmax"], cmap="coolwarm", title="PRISM normals"
)

### 2g) SSURGO soil survey (variable native, CONUS only)

USDA SSURGO aggregated from map-unit polygons: available water capacity, clay
fraction, saturated hydraulic conductivity, sand fraction. Survey-based rather
than model-predicted.

In [ ]:
plot_group(
    "ssurgo",
    ["ssurgo_awc", "ssurgo_clay", "ssurgo_ksat"],
    cmap="YlOrBr",
    title="SSURGO",
)

### 2h) FAO Harmonized World Soil Database (1 km native)

HWSD v2.0 harmonizes national soil surveys onto a common schema: classification
codes (WRB, FAO90), physical properties (bulk density, drainage, texture), and
derived metrics (AWC, root depth). Global coverage is what lets the released
model run outside CONUS, where SSURGO and POLARIS do not exist.

In [ ]:
plot_group(
    "fao_hwsd", ["BULK_DENSITY", "AWC", "TEXTURE_USDA"], cmap="YlOrBr", title="FAO HWSD"
)

### 2i) Land cover

NLCD (30 m), USDA Cropland Data Layer (30 m, modal crop type), C3S Land Cover
(300 m, LCCS classes), Global Land Cover 10 m, and JRC Global Surface Water
occurrence. These are categorical, so `reproject_to_ease2` resamples them with
nearest neighbour — `NEAREST_GROUPS` in that module is the list.

In [ ]:
from swapstress.features.reproject_to_ease2 import NEAREST_GROUPS

print("resampled with nearest neighbour:", ", ".join(sorted(NEAREST_GROUPS)))
plot_group(
    "landcover",
    ["nlcd", "cdl_crop_mode", "c3s_lccs_class_mode"],
    cmap="tab20",
    title="Land cover",
)

### 2j) Landsat 8/9 spectral bands (30 m native)

Growing-season and quarterly (Q1-Q4) mean and standard deviation of surface
reflectance bands B2-B7 plus thermal brightness temperature B10 — vegetation
phenology and surface energy balance seasonality.

In [ ]:
plot_group(
    "landsat_bands",
    ["B5_mean_gs", "B4_mean_gs", "B10_mean_gs"],
    cmap="YlGn",
    title="Landsat 8/9 growing-season composites",
)

### 2k) Sentinel-1 C-band SAR (10 m native)

Mean and standard deviation of GRD backscatter in VV and VH, plus the VH/VV
ratio. C-band backscatter responds to surface roughness, soil moisture, and
vegetation structure at once, which is what makes it informative and hard to
interpret in isolation.

In [ ]:
plot_group("sentinel1", ["VV_mean", "VH_mean"], cmap="gray", title="Sentinel-1 SAR")

## 3) Band counts

How many bands each group contributes. The released model does not use all of
them — `configs/train_9km_global_pruned.toml` keeps five groups — but the full
stack is what stage 05 has available.

In [ ]:
counts = bands_df.groupby("file").size().rename("bands").sort_values().reset_index()
counts["group"] = counts["file"].str.replace("_ease2.tif", "", regex=False)

fig, ax = plt.subplots(figsize=(8, 5), dpi=120)
ax.barh(counts["group"], counts["bands"], color="#607D8B", edgecolor="white")
for i, n in enumerate(counts["bands"]):
    ax.text(n + 0.5, i, str(n), va="center", fontsize=9)

ax.set_xlabel("Number of bands")
ax.set_title(f"Feature bands per file (total: {counts['bands'].sum()})")

fig.tight_layout()
out = os.path.join(OUT_DIR, "9k_feature_band_counts.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

In [ ]:
# The same bands regrouped the way the model sees them.
bands_df.groupby("group").size().sort_values(ascending=False).to_frame("bands")

## Next

`07_inference.ipynb` puts this stack and the daily SMAP rasters through the
trained model (stages 05-07).